# Run PCR Simulation 
in Sri Lanka using NHPP method 

In [ ]:
import os 

import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
import xarray as xr

from dotenv import load_dotenv
from pcr import helper, storm, slr, erosion, shoreline

load_dotenv()

In [ ]:
import importlib
importlib.reload(slr)
importlib.reload(erosion)
importlib.reload(shoreline)
importlib.reload(helper)

In [ ]:
# access data from CDS ARCO 
lon_bat = 82.0 # Sri Lanka
lat_bat = 7.5
time_slice = slice('1979-01-01', '2019-12-31')

# lazy load the ERA5 data
wave_geo_url = "https://arco.datastores.ecmwf.int/cadl-arco-geo-003/arco/reanalysis_era5_single_levels/wav/geoChunked.zarr"

# open the zarr object with xarray 
ds = xr.open_zarr(
    wave_geo_url, 
    consolidated=True, 
    storage_options={
        'headers': {'Authorization': f'Bearer {os.getenv('CDS-API-KEY')}'}
    }
)

# load in first location 
ds = ds.sel(
    longitude=lon_bat, 
    latitude=lat_bat,
    method='nearest'
).sel(time=time_slice).compute()

In [ ]:
era5_mapper = {'hs':'swh', 'dir':'mwd', 'tp':'mwp', 'time':'time'}

In [ ]:
hs, dir, tp, time = helper.era5_input(ds, {'hs':'swh', 'dir':'mwd', 'tp':'mwp', 'time':'time'})
# check the time key 
data_year = (ds.time[-1].dt.year - ds.time[0].dt.year).values

# storm detection 
ts_hs = 95 
ts_dur = 12.0 
ts_between = 48 

detected_storms, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur, ts_between) # identify independent storms 

# fit storm and gap 
fitted_storm = storm.fit_storm(detected_storms)
fitted_lambdas = storm.fit_lambda_gap(detected_storms)

# set up simulation 
date_start = np.datetime64('2000-01-01T00:00:00')
date_end = np.datetime64('2100-12-31T23:00:00')

nr_simulation = 100000
nr_batch = 1000
yearly_n_storm = np.ceil(detected_storms.shape[0] / data_year)

In [ ]:
synth_start = storm.gap_nhpp_thinning(
            T=t_days, 
            monthly_lambda=fitted_lambdas, 
            date_start=date_start, 
            duration=durs, 
            start_storm=storm_count
        )

len(synth_start)

In [ ]:
# function similar as generate_monsoon_ts

t_days = (date_end-date_start).item().days
t_years = (date_end.astype('datetime64[Y]') - date_start.astype('datetime64[Y]')).astype(int)

sim_count = 0
shoreline_stats = np.empty((t_years+1, nr_simulation))

while sim_count < nr_simulation: 
    n_sample =  yearly_n_storm * t_years * nr_batch

    hss, durs, dirs, tps = storm.generate(
        fitted_storm=fitted_storm, 
        sampling_size=n_sample, 
        oversample=0.1, 
        max_dur=np.max(detected_storms.duration)
    )

    # if possible, run the gap_nhpp_thinning for batch first 
    for i in range(nr_batch):

        storm_count = 0

        synth_start = storm.gap_nhpp_thinning(
            T=t_days, 
            monthly_lambda=fitted_lambdas, 
            date_start=date_start, 
            duration=durs, 
            start_storm=storm_count
        )

        storm_count_end = storm_count + len(synth_start)

        synth_hs =  hss[storm_count:storm_count_end]
        synth_direction =  dirs[storm_count:storm_count_end]
        synth_duration = durs[storm_count:storm_count_end]
        synth_tp = tps[storm_count:storm_count_end]
        synth_end =  synth_start + (synth_duration / 24) 
        synth_gap = synth_start - np.roll(synth_end, 1)
        synth_gap[0] = 0.0

        storm_count = storm_count_end

        # synthetic_storm = pd.DataFrame({
        #     'hs': synth_hs,
        #     'direction': synth_direction,
        #     'duration': synth_duration,
        #     'tp': synth_tp,
        #     'day_start': synth_start,
        #     'day_end': synth_end, 
        #     'gap': synth_gap
        # })

        synth_slr = slr.vector_simulate_slr(
            day_start=synth_start, 
            date_start=date_start, 
            scenario='0', 
            wl0=0
        )

        _, synth_erosion = erosion.vector_mendoza(
            hss=synth_hs, 
            tps=synth_tp, 
            durs=synth_duration
        )

        synth_recovery = shoreline.vector_calculate_recovery(
            gaps=synth_gap, 
            rec_rate=7/365
        )

        synth_retreat = shoreline.vector_calculate_slr_retreat(
            slrs=synth_slr,
            m=0.024
        )

        track_time, track_shoreline_change, track_shoreline_position = shoreline.vector_track_shoreline(
            day_start=synth_start, 
            day_end=synth_end, 
            recovery=synth_recovery,
            retreat=synth_retreat, 
            erosion=synth_erosion
        )

        row = shoreline.vector_get_annual_statistics(
            track_time=track_time, 
            shoreline_position=track_shoreline_position, 
            kind='min', 
            date_start=date_start)


        try: 
            shoreline_stats[:, sim_count] = row.flatten()
        except: 
            sim_count -= 1

        sim_count += 1

In [ ]:
pd.Series(helper.date_add_days(date_start, track_time))

In [ ]:
pd.DataFrame(shoreline_stats).to_csv('../data/output/sl_out_thinning.csv')


In [ ]:
shoreline_stats